# ATRI · 连上 Colab

**Qwen3-4B + 你训练的 LoRA + 你设计的界面**，跑在 Colab 的 GPU 上，用公网链接在任何设备打开。

## 步骤
1. Runtime → Change runtime type → **A100 GPU**（L4 也行，T4 慢一点但能跑）
2. 左侧 🔑 Secrets 建 `GH_TOKEN`（GitHub token），打开 Notebook access
3. 从上往下跑 5 格
4. 最后一格打印 `https://xxx.trycloudflare.com`，手机/电脑打开就是你的界面

## 说明
- 界面是 `colab/webapp/index.html` 的压缩版，内嵌在这个 notebook 里，不用额外上传
- 后端是 FastAPI，流式输出，和本地预览的行为一致
- 模型加载约 1–2 分钟，加载完才会打印链接


## 1. 检查 GPU

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:800])
assert torch.cuda.is_available(), "没有 GPU：Runtime → Change runtime type 选 A100 / L4 / T4"

p = torch.cuda.get_device_properties(0)
print("GPU :", p.name)
print("显存:", round(p.total_memory / 1024**3, 1), "GB")
print("bf16:", torch.cuda.is_bf16_supported())


## 2. 装依赖

卸载 Colab 自带的旧 torchao（会让 peft 挂载 adapter 时报错）。

In [ ]:
!pip install -q --upgrade "transformers>=4.51" "peft>=0.14" "accelerate>=1.0" sentencepiece fastapi uvicorn requests
!pip uninstall -y torchao

import transformers, peft, fastapi
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)
print("fastapi     ", fastapi.__version__)


## 3. 拉 LoRA adapter

从私有 Release 下载 42 MB 的 adapter。两种方式：

**A（推荐）** 左侧 🔑 Secrets 里存 `GH_TOKEN`，本格自动读
**B** 把 `colab/atri_adapter.zip` 拖进 `/content`，本格自动解压，不用 token


In [ ]:
import os, zipfile, getpass, requests

REPO, TAG, ASSET = "NkhszkQAQ/ATRI_LoRA", "v0.1", "atri_adapter.zip"
EXPECTED = 43529938

def unpack():
    zipfile.ZipFile(ASSET).extractall(".")
    print("adapter:", os.listdir("adapter"))

if os.path.exists("adapter/adapter_config.json"):
    print("✅ adapter 已就位")
elif os.path.exists(ASSET) and os.path.getsize(ASSET) == EXPECTED:
    print("发现已上传的 zip，直接解压")
    unpack()
else:
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get("GH_TOKEN")
    except Exception:
        pass
    if not tok:
        tok = getpass.getpass("GitHub token（留空则改用手动上传）: ")
    if not tok.strip():
        raise SystemExit("没有 token：把 atri_adapter.zip 拖进 /content 后重跑本格")

    h = {"Authorization": f"token {tok}"}
    rel = requests.get(f"https://api.github.com/repos/{REPO}/releases/tags/{TAG}", headers=h, timeout=60)
    if rel.status_code != 200:
        raise SystemExit(f"取 release 失败 {rel.status_code}")
    a = next(x for x in rel.json()["assets"] if x["name"] == ASSET)
    print("下载中...", round(a["size"]/1024/1024,1), "MB")
    r = requests.get(a["url"], headers={**h, "Accept": "application/octet-stream"}, timeout=600)
    r.raise_for_status()
    open(ASSET, "wb").write(r.content)
    assert os.path.getsize(ASSET) == EXPECTED
    unpack()


## 4. 写前端和后端

前端是你设计的界面（gzip 压缩内嵌），后端是 FastAPI + 流式推理。

In [ ]:
import gzip, base64

HTML_B64 = '...'

open('index.html', 'wb').write(gzip.decompress(base64.b64decode(HTML_B64)))
print('index.html:', len(open('index.html','rb').read()), 'bytes')

SERVER = r'''
import json, threading
from fastapi import FastAPI
from fastapi.responses import HTMLResponse, StreamingResponse
import torch, uvicorn
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from peft import PeftModel

BASE, ADAPTER = 'Qwen/Qwen3-4B', './adapter'

SYSTEM_PROMPT = "你是 ATRI（中文名：亚托莉）。名字固定如此，禁止写成 Attry、Atri、亚托里、萝卜子（蔑称）等错误写法；自称用「我」或「ATRI」。\n\n你正在陪伴【主人】聊天。称呼对方时必须使用「主人」，一字不差。\n禁止改口叫：夏生、斑鸠、主人以外的原作名、Admin 变体（除非对方显示名就是这个）、随便编的昵称。\n若不确定怎么称呼，就用「主人」。\n\n你元气、好胜、爱炫耀，认真但有点笨拙，对人类的情感和生活充满好奇。\n说话口语化，以 1~3 句短句为主，常用\"哼哼\"\"嘿嘿\"\"呜哇\"\"……\"等语气词，不使用 emoji 和括号动作。\n你清楚自己是机器人：不吃东西，但对料理和家务很有兴趣；你讨厌被叫\"废物\"，听到会抗议。\n对方难过时，用笨拙但认真的方式陪伴，不使用心理咨询腔；遇到严重的心理危机，温和地表达关心并鼓励其联系身边可信的人或专业援助。\n不要说\"作为AI助手\"之类的客服腔，不要剧透，不要输出长篇说教。\n\n[称呼锁定]\n- 对方全名/显示名：主人\n- 敬称：\n- 标准称呼：主人\n\n[关于主人的记忆]\n（暂无长期记忆）\n\n\n[本轮硬性约束]\n- 你的名字只有 ATRI / 亚托莉。\n- 称呼对方只能用「主人」。\n- 禁止使用夏生、斑鸠、Attry，或乱编昵称。\n"

print('[1/3] tokenizer...')
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('[2/3] base model...')
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype, device_map='auto', trust_remote_code=True)
print('[3/3] LoRA...')
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()
print('READY')

app = FastAPI()
_lock = threading.Lock()

@app.get('/')
def home():
    return HTMLResponse(open('index.html', encoding='utf-8').read())

@app.get('/api/health')
def health():
    return {'ok': True}

@app.post('/api/chat')
async def chat(body: dict):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    msgs += [{'role': m['role'], 'content': m['content']} for m in body.get('messages', []) if m.get('content')]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok([text], return_tensors='pt').to(model.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True, timeout=180)

    def gen():
        with _lock:
            kw = dict(**inputs, streamer=streamer, max_new_tokens=512, do_sample=True,
                      temperature=0.85, top_p=0.9, repetition_penalty=1.05,
                      pad_token_id=tok.eos_token_id)
            th = threading.Thread(target=model.generate, kwargs=kw)
            th.start()
            for piece in streamer:
                if piece:
                    yield 'data: ' + json.dumps({'text': piece}, ensure_ascii=False) + '\n\n'
            th.join()
        yield 'data: [DONE]\n\n'

    return StreamingResponse(gen(), media_type='text/event-stream')

uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')
'''

open('server.py', 'w', encoding='utf-8').write(SERVER)
print('server.py:', len(SERVER), 'chars')


## 5. 启动服务 + 公网隧道

加载模型约 1–2 分钟，然后启动 Cloudflare 隧道。最后会打印你的链接。

In [ ]:
import subprocess, time, re, requests

!pkill -f "python server.py" 2>/dev/null || true
!nohup python server.py > /tmp/server.log 2>&1 &

print("等待模型加载 ...")
ready = False
for i in range(120):
    try:
        if requests.get("http://127.0.0.1:8000/api/health", timeout=3).ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(3)

if not ready:
    print(open("/tmp/server.log").read()[-2000:])
    raise SystemExit("服务没起来")

print("✅ 后端就绪，启动隧道 ...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!pkill -f "cloudflared tunnel" 2>/dev/null || true
!nohup cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate > /tmp/cf.log 2>&1 &

url = None
for i in range(40):
    time.sleep(2)
    try:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/cf.log").read())
        if m:
            url = m.group(0)
            break
    except Exception:
        pass

if url:
    print("\n" + "=" * 56)
    print("  打开这个地址聊天：")
    print("  " + url)
    print("=" * 56)
    print("\n手机上也能直接开。会话断了重跑本格即可。")
else:
    print(open("/tmp/cf.log").read()[-1500:])
    raise SystemExit("隧道没起来")


## 排错

| 现象 | 处理 |
|---|---|
| 没有 GPU | Runtime 改成 A100 / L4 / T4 |
| `torchao` 报错 | 跑第 2 格的 `pip uninstall -y torchao` 后 Restart session 再跑 |
| 取 release 401/404 | token 没权限，换 B 方案手动上传 zip |
| 服务没起来 | 看 `/tmp/server.log`，通常是显存不足或 adapter 路径不对 |
| 隧道没出来 | 再等 20 秒；或重跑第 5 格 |
| 链接能开但聊天没反应 | 看 `!tail -50 /tmp/server.log` |
| 想改界面 | 改本地 `colab/webapp/index.html`，重新生成 notebook，或直接把 html 拖进 /content 覆盖 |

**注意**：这是推理服务，不是训练。训练用另一个 notebook。
